In [28]:
import re
from collections import Counter

data = [
    "Ths is a smple txt with errrs.",
    "This is a clean sentence.",
    "0CR scann3d d0cum3nt with nois3",
    "lI l IlI random garbage xqztr",
    "The quick brown fox jumps over the lazy dog."
]

In [29]:
# Layer1: Sanity check

def is_mostly_text(chunk, threshold=0.7):
    letters = sum(c.isalpha() for c in chunk)
    return letters / max(len(chunk), 1) >= threshold

def has_reasonable_word_lengths(chunk, min_avg=2, max_avg=12):
    words = chunk.split()
    if not words:
        return False
    avg_len = sum(len(w) for w in words) / len(words)
    return min_avg <= avg_len <= max_avg

def passes_basic_checks(chunk):
    return (
        is_mostly_text(chunk) and
        has_reasonable_word_lengths(chunk)
    )

filtered_basic = [c for c in data if passes_basic_checks(c)]

print("After Layer 1:", filtered_basic)

After Layer 1: ['Ths is a smple txt with errrs.', 'This is a clean sentence.', '0CR scann3d d0cum3nt with nois3', 'lI l IlI random garbage xqztr', 'The quick brown fox jumps over the lazy dog.']


In [30]:
# Layer 2: Language check
# Uses lingua (accurate on noisy text; works fully offline).
import subprocess
import sys
from functools import lru_cache
from lingua import Language, LanguageDetectorBuilder



@lru_cache(maxsize=1)
def _language_detector():
    # Full spoken-language set so non-English lines get a competing hypothesis
    return LanguageDetectorBuilder.from_all_spoken_languages().build()


def passes_english_language_check(text, min_chars_for_id=20):
    """
    Keep chunks that are detected as English. Very short snippets are skipped
    (unreliable); they are kept if they passed Layer 1.
    """
    t = text.strip()
    if len(t) < min_chars_for_id:
        return True
    lang = _language_detector().detect_language_of(t)
    return lang == Language.ENGLISH


filtered_english = [c for c in filtered_basic if passes_english_language_check(c)]

print("After English filter:", filtered_english)

After English filter: ['Ths is a smple txt with errrs.', 'This is a clean sentence.', 'The quick brown fox jumps over the lazy dog.']


In [31]:
# Layer 3 - Spell Correction

from difflib import get_close_matches

def correct_word(word, vocab):
    matches = get_close_matches(word, vocab, n=1, cutoff=0.8)
    return matches[0] if matches else word

def correct_text(text, vocab):
    words = text.split()
    corrected = [correct_word(w.lower(), vocab) for w in words]
    return " ".join(corrected)

corrected = [correct_text(c, ENGLISH_WORDS) for c in filtered_english]

print("After correction:", corrected)

After correction: ['this is a simple text with errors', 'this is a clean sentence', 'the quick brown fox jumps over the lazy dog']


In [32]:
# Layer 4 - Final decision logic
def score_text(text):
    return {
        "english_ratio": english_ratio(text),
        "length": len(text)
    }

def final_filter(text):
    score = score_text(text)
    
    if score["english_ratio"] < 0.6:
        return False
    if score["length"] < 10:
        return False
    
    return True

final_dataset = [c for c in corrected if final_filter(c)]

print("Final dataset:", final_dataset)

Final dataset: ['this is a simple text with errors', 'this is a clean sentence', 'the quick brown fox jumps over the lazy dog']
